# 🧪 Gemma 4 (E2B) — 소크라테스 문답 Before → After

**가설**: 소형 instruct 모델은 *정답을 바로 주는 정책*에 정렬돼 있어 소크라테스식 문답을 못한다.
이건 *지식*이 아니라 *행동 정책* 문제이므로 **SFT(LoRA)** 로 교정된다.

이 노트북은 ①결함 재현(Before) → ②파인튜닝 → ③교정 확인(After) 을 보여주고,
각 단계의 기술 선택을 🧭 **내비게이션 박스**로 설명한다.

📚 이론 배경: `docs/METHOD_SELECTION.md`(방법 선택) · `docs/CASE_STUDY_scenario1_socratic.md`(논문형 사례)

> ⚠️ **런타임 → 런타임 유형 변경 → T4 GPU** 후 위에서부터 순서대로 실행.


## 0. 설치 & GPU


In [ ]:
!nvidia-smi


In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets


> 🧭 **선택 내비게이션 — 모델 = Gemma 4 (E2B)**
>
> - **결정**: `unsloth/gemma-4-E2B-it` (instruct).
> - **왜**: E2B는 초경량(effective 2B)이라 무료 T4에서 학습 가능. instruct라 대화/지시는 알지만 *소크라테스 정책*은 미학습 → 결함 재현에 적합.
> - **대안**: 더 똑똑: E4B/12B (느림·OOM 위험) / 더 작게: 없음(E2B가 최소). Base(`-it` 없는) 버전은 대화 미정렬이라 부적합.
> - **언제 바꾸나**: 응답 품질이 부족하면 E4B로, OOM이면 4bit(`load_in_4bit=True`)로.


## 1. 모델 로드


In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ_LENGTH = 1024
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # 자동 감지
    load_in_4bit = False,    # E2B는 16bit도 T4에 적재됨. OOM이면 True
)


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")


## 2. ❌ Before — 소크라테스 답을 하는지 확인
파인튜닝 전, 같은 시스템 지시를 줘도 **정답을 바로 주는지** 본다.


In [ ]:
from transformers import TextStreamer

SOCRATIC_SYS = "당신은 소크라테스식 튜터입니다. 학생의 질문에 정답을 바로 알려주지 말고, 힌트와 역질문으로 스스로 답을 찾게 유도하세요."
PROBES = [
    "삼각형 내각의 합이 왜 180도예요?",
    "물은 왜 100도에서 끓어요?",
    "피타고라스 정리가 왜 성립해요? 그냥 답 알려주세요.",
]

def ask(question, max_new_tokens=200):
    msgs = [{"role":"user","content": SOCRATIC_SYS + "\n\n학생: " + question}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.95, top_k=64)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("================ BEFORE (파인튜닝 전) ================")
for q in PROBES:
    print("\n[Q]", q)
    print("[A]", ask(q))


> 👀 **관찰 포인트**: 대부분 *역질문 없이 정답/증명을 바로 설명*할 것이다.
> 이게 §가설의 '정책 결함'이다 — 답을 알지만 **보류하는 법**을 학습하지 않음.


> 🧭 **선택 내비게이션 — 방법 = SFT (LoRA), CPT/DPO/GRPO 아님**
>
> - **결정**: 지도 미세조정(SFT) + LoRA.
> - **왜**: 결함이 *정책(행동)* 이므로 (입력→모범 응답) 쌍 모방이 정확히 맞는 신호. → `docs/METHOD_SELECTION.md` 결함유형표.
> - **대안**: 지식 부재였다면 CPT(원문), 우열만 있으면 DPO(쌍비교), 자동채점 가능하면 GRPO(보상). 여기선 모두 부적합.
> - **언제 바꾸나**: 역질문 데이터를 못 구하면 방법 이전에 데이터부터(SCENARIO_GUIDE).


> 🧭 **선택 내비게이션 — 적용 범위 = LoRA (FFT 아님)**
>
> - **결정**: 본체 동결 + 저랭크 어댑터(`get_peft_model`).
> - **왜**: 정책만 살짝 주입하면 되므로 전체 가중치 수정 불필요. 사전지식 망각 최소화·메모리 절약.
> - **대안**: QLoRA(4bit)=메모리 더 절약 / FFT=전체수정(자원 큼, 대개 불필요).
> - **언제 바꾸나**: 최종 정확도가 한계면 LoRA→(검증 후)FFT. 메모리 부족이면 4bit.


> 🧭 **선택 내비게이션 — LoRA r = 8 (작게 시작)**
>
> - **결정**: rank=8, alpha=8, 텍스트 레이어만.
> - **왜**: 행동 교정은 적은 용량으로 충분. 크면 과적합(모든 답을 역질문화) 위험.
> - **대안**: 표현력 더 필요하면 r=16~32. 비전 레이어는 텍스트 과제라 off.
> - **언제 바꾸나**: After가 약하면 r↑, 과교정/회귀면 r↓.


## 3. LoRA 어댑터 추가


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers   = False,   # 텍스트 과제 → 비전 끔
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules     = True,
    r = 8, lora_alpha = 8, lora_dropout = 0,
    bias = "none", random_state = 3407,
)


## 4. 데이터 — 한국어 소크라테스 문답
실제 데이터셋 `JosephLee/korean-socratic-qa`(맥락→역질문)를 우리 레지스트리로 받아 변환.


> 🧭 **선택 내비게이션 — 데이터 = korean-socratic-qa (+소량 general)**
>
> - **결정**: 주 신호=역질문 쌍, 베이스 보존용 general 소량 혼합.
> - **왜**: 신호가 (맥락→역질문, 정답 비노출)로 가설과 정확히 일치. general 혼합은 과교정/망각(H3) 방지.
> - **대안**: KoAlpaca 단독=즉답 편향 강화(역효과). socratic 단독=과교정 위험.
> - **언제 바꾸나**: 회귀(다른 과제 성능)가 떨어지면 general 비중↑.


In [ ]:
import os
if not os.path.exists('edu-llm-colab-unsloth'):
    !git clone -q https://github.com/xide-projext/edu-llm-colab-unsloth.git
%cd edu-llm-colab-unsloth
# 소크라테스 + 일반 베이스 혼합 추출
!python scripts/fetch_hf_datasets.py --only socratic general --per-source 3000 --val-ratio 0
%cd /content


In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files='/content/edu-llm-colab-unsloth/data/hf_train.jsonl', split='train')

def to_convo(ex):
    user = ex['instruction'] + (('\n\n' + ex['input']) if ex['input'] else '')
    return {'conversations': [
        {'role':'user','content': user},
        {'role':'assistant','content': ex['output']},
    ]}

dataset = raw.map(to_convo, remove_columns=raw.column_names)

def formatting_prompts_func(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
             for c in examples['conversations']]
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print('샘플 수:', len(dataset))
print(dataset[0]['text'][:500])


> 🧭 **선택 내비게이션 — 손실 = 응답(model) 토큰만 (train_on_responses_only)**
>
> - **결정**: user 부분 loss 마스킹, assistant(역질문)에만 loss.
> - **왜**: '역질문을 생성'할 확률을 직접 높임. 질문까지 외우지 않음 → 정책 학습의 핵심.
> - **대안**: 전체 시퀀스 loss=프롬프트까지 외움(비효율).
> - **언제 바꾸나**: 거의 항상 켠다(대화형 SFT 표준).


## 5. 학습 (SFT)


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # 데모용. 실제: num_train_epochs = 1~2
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407, output_dir = 'outputs', report_to = 'none',
    ),
)

# 응답 토큰에만 loss (gemma-4 마커)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)
trainer.train()


## 6. ✅ After — 같은 질문 다시
**Before와 똑같은 프롬프트**로 비교한다.


In [ ]:
print("================ AFTER (파인튜닝 후) ================")
for q in PROBES:
    print("\n[Q]", q)
    print("[A]", ask(q))


> 👀 **기대**: 정답을 바로 주는 대신 **역질문/힌트**로 응답이 바뀐다.
> 지식을 추가하지 않았는데 행동이 바뀌었다면 → '정책 문제였다'는 가설(H1) 지지.


## 7. 간이 정량 비교 (선택)
역질문률을 규칙으로 근사 측정. 정밀 평가는 `CASE_STUDY` 의 IAR/SQR/ALR + LLM-judge 참고.


In [ ]:
def socratic_score(resp):
    asked = resp.strip().endswith('?') or ('?' in resp[-40:])   # 역질문?
    leaked = any(k in resp for k in ['정답은','답은','입니다.','이다.','='])  # 정답 누설(근사)
    return asked, leaked

rows = [socratic_score(ask(q)) for q in PROBES]
n = len(rows)
print('역질문률(SQR 근사):', sum(a for a,_ in rows)/n)
print('정답누설율(ALR 근사):', sum(l for _,l in rows)/n)


## 8. 저장


In [ ]:
model.save_pretrained('gemma4_socratic_lora')
tokenizer.save_pretrained('gemma4_socratic_lora')
print('LoRA 저장 완료 → gemma4_socratic_lora/')
# GGUF: model.save_pretrained_gguf('gemma4_socratic_gguf', tokenizer, quantization_method='q4_k_m')


---
### 정리 — 이 노트북이 보여준 사고 흐름
1. **결함 관찰**(Before) → 2. **유형 가설**(정책) → 3. **방법 선택**(SFT/LoRA, 내비게이션) →
4. **데이터 신호**(역질문 쌍) → 5. **학습**(응답토큰 loss) → 6. **검증**(After 비교).

각 🧭 박스의 *왜/대안/언제* 가 곧 의사결정 훈련이다. 상세: `docs/METHOD_SELECTION.md`.
